# Module 4 - Week 6, Topic 3: Decision Trees
## Live Demo Notebook

**Scenario:** A Nigerian bank needs a credit risk model that the compliance team can audit.
The regulator requires that rejected applicants receive a clear explanation.

We build a **Decision Tree** - the most interpretable ML classifier.

---

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier, plot_tree, export_text
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

sns.set_theme(style='whitegrid')

df = pd.read_csv('credit_risk.csv')
print('Shape:', df.shape)
print(f'High risk rate: {df["high_risk"].mean():.1%}')
df.head()

In [ ]:
FEATURES = ['income', 'credit_score', 'debt_ratio', 'employment_years', 'num_late_payments']
TARGET   = 'high_risk'

X = df[FEATURES]
y = df[TARGET]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
print(f'Training: {len(X_train)} | Test: {len(X_test)}')

---
## Part 1 - Three Trees: Underfit, Overfit, Good Fit

In [ ]:
models = {
    'Underfit  (depth=1)':    DecisionTreeClassifier(max_depth=1,    random_state=42),
    'Good fit  (depth=4)':    DecisionTreeClassifier(max_depth=4,    random_state=42),
    'Overfit   (depth=None)': DecisionTreeClassifier(max_depth=None, random_state=42),
}

print(f'{"Model":<30} {"Train":>8} {"Test":>8} {"Gap":>8} {"Verdict":>12}')
print('-' * 72)
for name, m in models.items():
    m.fit(X_train, y_train)
    tr = m.score(X_train, y_train)
    te = m.score(X_test,  y_test)
    gap = tr - te
    verdict = 'UNDERFIT' if tr < 0.65 else ('OVERFIT' if gap > 0.15 else 'GOOD FIT')
    print(f'{name:<30} {tr:>7.1%} {te:>8.1%} {gap:>7.1%} {verdict:>12}')

---
## Part 2 - Visualise the Good-Fit Tree

In [ ]:
# Train the good-fit model
dt = DecisionTreeClassifier(
    max_depth=4,
    min_samples_leaf=5,
    random_state=42
)
dt.fit(X_train, y_train)

# Text representation - easy to read and share with non-technical stakeholders
print('=== Decision Tree Rules (text format) ===')
print()
print(export_text(dt, feature_names=FEATURES))

In [ ]:
# Visual representation
fig, ax = plt.subplots(figsize=(18, 8))
plot_tree(
    dt,
    feature_names=FEATURES,
    class_names=['Low Risk', 'High Risk'],
    filled=True,
    rounded=True,
    fontsize=9,
    ax=ax,
    impurity=True,
)
ax.set_title('Credit Risk Decision Tree (max_depth=4, min_samples_leaf=5)',
             fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('credit_risk_tree.png', dpi=120, bbox_inches='tight')
plt.show()
print('Tree saved as credit_risk_tree.png')

---
## Part 3 - Evaluate the Model

In [ ]:
y_pred = dt.predict(X_test)
print(f'Test accuracy: {accuracy_score(y_test, y_pred):.2%}')
print()
print(classification_report(y_test, y_pred, target_names=['Low Risk', 'High Risk']))

In [ ]:
cm = confusion_matrix(y_test, y_pred)

fig, ax = plt.subplots(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=['Pred: Low Risk', 'Pred: High Risk'],
            yticklabels=['Actual: Low Risk', 'Actual: High Risk'],
            ax=ax, linewidths=0.5)
ax.set_title('Confusion Matrix - Credit Risk Decision Tree',
             fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()

---
## Part 4 - Feature Importance

In [ ]:
importance = pd.Series(
    dt.feature_importances_,
    index=FEATURES
).sort_values(ascending=True)

fig, ax = plt.subplots(figsize=(8, 4))
importance.plot(kind='barh', ax=ax, color='steelblue', edgecolor='white')
ax.set_title('Feature Importance - Credit Risk Decision Tree',
             fontsize=12, fontweight='bold')
ax.set_xlabel('Importance Score')
plt.tight_layout()
plt.show()

print('Most important features for credit risk:')
for feat, imp in importance.sort_values(ascending=False).items():
    bar = '#' * int(imp * 40)
    print(f'  {feat:<25} {imp:.3f}  {bar}')

---
## Part 5 - CBN Compliance: Explaining a Rejection

In [ ]:
# A specific loan applicant
applicant = pd.DataFrame({
    'income':            [180000],
    'credit_score':      [590],
    'debt_ratio':        [0.72],
    'employment_years':  [2],
    'num_late_payments': [4],
})

prediction = dt.predict(applicant[FEATURES])
probability = dt.predict_proba(applicant[FEATURES])[0]

print('=== Loan Application Review ===')
print(f'Applicant: income=NGN {applicant.income.iloc[0]:,}')
print(f'           credit_score={applicant.credit_score.iloc[0]}')
print(f'           debt_ratio={applicant.debt_ratio.iloc[0]}')
print(f'           employment_years={applicant.employment_years.iloc[0]}')
print(f'           num_late_payments={applicant.num_late_payments.iloc[0]}')
print()
result = 'HIGH RISK - Credit Application Declined' if prediction[0] else 'LOW RISK - Credit Application Approved'
print(f'Decision: {result}')
print(f'Confidence: Low Risk={probability[0]:.1%}, High Risk={probability[1]:.1%}')
print()
print('Audit trail (decision path through the tree):')
# Get the decision path
node_indicator = dt.decision_path(applicant[FEATURES])
leaf_id = dt.apply(applicant[FEATURES])
print(f'  Applicant passes through {node_indicator.nnz} nodes')
print(f'  Final leaf node: {leaf_id[0]}')
print()
print('This decision path can be printed and provided to the applicant')
print('and the CBN in compliance with responsible lending guidelines.')

---
## Part 6 - Tree Instability Demo

In [ ]:
# Show how the tree changes with different random seeds (instability)
print('Decision Tree root split with different random states:')
print()

for seed in [0, 10, 42, 99, 200]:
    dt_temp = DecisionTreeClassifier(max_depth=4, min_samples_leaf=5, random_state=seed)
    dt_temp.fit(X_train, y_train)
    test_acc = dt_temp.score(X_test, y_test)
    root_feature = FEATURES[dt_temp.tree_.feature[0]]
    root_threshold = dt_temp.tree_.threshold[0]
    print(f'  seed={seed:>3}: root split = "{root_feature} <= {root_threshold:.2f}" | test acc = {test_acc:.2%}')

print()
print('Different seeds can produce different root splits and accuracy scores.')
print('This INSTABILITY is a key weakness of single Decision Trees.')
print('Topic 4 (Random Forest) solves this by averaging 100 trees.')

---
## Summary

| What we did | Key insight |
|---|---|
| Three trees (under/good/overfit) | Depth=4 with min_samples_leaf=5 gives the best test accuracy |
| Text tree | Rules are directly readable - printable for compliance |
| Visual tree | `plot_tree()` shows the full decision logic graphically |
| Confusion matrix | Identified false positives and false negatives |
| Feature importance | Which features drive credit risk most |
| Compliance trace | The decision path satisfies CBN audit requirements |
| Instability demo | Different seeds → different trees - single tree is unstable |

**Next - Topic 4:** Random Forests. We fix the instability by training 100 trees and averaging - same features, much more robust predictions.